# DanceTrack – step 0: extract the dataset zips
Download `train1.zip`, `train2.zip`, `val.zip`, `test1.zip`, `test2.zip` from the DanceTrack page into `data/DanceTrack/`, then run this cell.

In [ ]:
# ── Extract DanceTrack zips that are already in data/DanceTrack ──
# Safe to re-run: finished zips are skipped. Takes ~10-20 min (11 GB).
import shutil, zipfile
from pathlib import Path

ROOT  = Path(r"data/DanceTrack")
FILES = {'val.zip': 'val', 'train1.zip': 'train', 'train2.zip': 'train',
         'test1.zip': 'test', 'test2.zip': 'test'}          # test zips are used only if present

for name, split in FILES.items():
    zpath = ROOT / name
    done_flag = ROOT / f'_{name}.done'
    if not zpath.exists():
        if split != 'test':
            print(f'{name}: NOT FOUND in {ROOT}')
        continue
    if done_flag.exists():
        print(f'{name}: already extracted'); continue
    print(f'Extracting {name} …')
    tmp = ROOT / f'_tmp_{zpath.stem}'
    shutil.rmtree(tmp, ignore_errors=True)                  # clean up a half-finished earlier attempt
    with zipfile.ZipFile(zpath) as z:
        z.extractall(tmp)
    # move every video folder (one that contains img1\) to ROOT\<split>\<video>
    dst = ROOT / split; dst.mkdir(exist_ok=True)
    seq_dirs = sorted({p.parent for p in tmp.rglob('img1') if p.is_dir()})
    n = 0
    for seq in seq_dirs:
        if not (dst / seq.name).exists():
            shutil.move(str(seq), str(dst / seq.name)); n += 1
    shutil.rmtree(tmp, ignore_errors=True)
    done_flag.touch()
    print(f'  {name}: {n} videos -> {dst}')

print()
for split in ('train', 'val', 'test'):
    d = ROOT / split
    if d.exists():
        seqs = [s for s in d.iterdir() if (s / 'img1').is_dir()]
        gt = sum((s / 'gt' / 'gt.txt').exists() for s in seqs)
        print(f'{split:5s}: {len(seqs):3d} videos ({gt} with gt)')
print('Expected: train 40, val 25 (test 35 later).')
print('When the counts are right you may delete the .zip files to free ~11 GB.')